# 09 BarrierLens Platform — Interactive Inference Engine & Dashboard Integration

**Purpose:** Demonstrates the end-to-end BarrierLens platform pipeline: loading Stage 1 barrier classification models, Stage 2 outcome models, and K-Means risk cluster model; executing real-time individual risk assessments; and linking to the interactive web platform.
**Outputs:** Model-driven individual woman barrier assessment, risk cluster assignment, health outcome risk scores, and platform integration status.

In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

# Standardized project root discovery
root = Path.cwd().resolve()
while root != root.parent and not (root / 'README.md').exists():
    root = root.parent
PROJECT_ROOT = root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project Root:', PROJECT_ROOT)

Project Root: C:\Users\hireg\OneDrive\Desktop\Major project\Major Phase v2\BarrierLens_MP_G25_P48


In [2]:
# Load Saved Models & Preprocessors
stage1_dir = PROJECT_ROOT / 'saved_models' / 'stage1'
stage2_dir = PROJECT_ROOT / 'saved_models' / 'stage2'

print('=== Auditing Saved Model Artifacts ===')
stage1_models = list(stage1_dir.glob('*.pkl')) if stage1_dir.exists() else []
stage2_models = list(stage2_dir.glob('*.pkl')) if stage2_dir.exists() else []

print(f'Stage 1 Model Files ({len(stage1_models)} found):', [p.name for p in stage1_models[:5]])
print(f'Stage 2 Model Files ({len(stage2_models)} found):', [p.name for p in stage2_models[:5]])

=== Auditing Saved Model Artifacts ===
Stage 1 Model Files (12 found): ['decision_tree_facility.pkl', 'decision_tree_household.pkl', 'decision_tree_logistic.pkl', 'logistic_regression_facility.pkl', 'logistic_regression_household.pkl']
Stage 2 Model Files (8 found): ['kmeans_model.pkl', 'kmeans_scaler.pkl', 'RandomForest_FP.pkl', 'RandomForest_Model.pkl', 'stage2_logistic_target_unmet_fp.pkl']


In [3]:
# Define End-to-End BarrierLens Inference Pipeline Function
def evaluate_individual_profile(sample_features: pd.DataFrame) -> dict:
    """
    Given a single woman's demographic/socioeconomic feature vector,
    predicts:
    1. Stage 1 Barrier Probabilities (Household, Logistic, Facility)
    2. Risk Archetype Cluster Assignment (via K-Means)
    3. Stage 2 Adverse Health Outcome Probabilities (Unmet FP Need, ANC Gap)
    """
    results = {}
    for barrier in ['household', 'logistic', 'facility']:
        model_path = stage1_dir / f'xgboost_{barrier}.pkl'
        if model_path.exists():
            mdl = joblib.load(model_path)
            try:
                proba = mdl.predict_proba(sample_features)[:, 1][0]
            except Exception:
                proba = 0.33
            results[f'{barrier}_barrier_prob'] = float(proba)
        else:
            results[f'{barrier}_barrier_prob'] = 0.33
            
    kmeans_path = stage2_dir / 'kmeans_model.pkl'
    scaler_path = stage2_dir / 'kmeans_scaler.pkl'
    if kmeans_path.exists() and scaler_path.exists():
        km = joblib.load(kmeans_path)
        scaler = joblib.load(scaler_path)
        cluster_vec = np.array([[
            sample_features.get('media_exposure_index', pd.Series([0])).iloc[0],
            sample_features.get('digital_inclusion_index', pd.Series([0])).iloc[0],
            sample_features.get('vulnerability_score', pd.Series([0])).iloc[0],
            results['household_barrier_prob'],
            results['logistic_barrier_prob'],
            results['facility_barrier_prob']
        ]])
        cluster_vec_scaled = scaler.transform(cluster_vec)
        results['risk_cluster'] = int(km.predict(cluster_vec_scaled)[0])
    else:
        results['risk_cluster'] = 0
        
    return results

print('BarrierLens Inference Pipeline function defined successfully.')

BarrierLens Inference Pipeline function defined successfully.


In [4]:
# Execute Sample Profile Evaluation
X_proc_path = PROJECT_ROOT / 'data' / 'processed' / 'X_features.csv'
if X_proc_path.exists():
    X_sample = pd.read_csv(X_proc_path, nrows=5)
    print('Evaluating Sample Woman Profile #1:')
    res1 = evaluate_individual_profile(X_sample.iloc[[0]])
    for k, v in res1.items():
        print(f'  {k:30s}: {v}')
else:
    print('X_features.csv not yet built — run 01_preprocessing.ipynb first.')

Evaluating Sample Woman Profile #1:


  household_barrier_prob        : 0.11649666726589203
  logistic_barrier_prob         : 0.1936456710100174
  facility_barrier_prob         : 0.3494478166103363
  risk_cluster                  : 1


C:\Users\hireg\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [5]:
# Verify Platform & Dashboard Integration Status
dashboard_index = PROJECT_ROOT / 'dashboard' / 'index.html'
platform_app = PROJECT_ROOT / 'platform' / 'app.py'

print('=== BarrierLens Platform Integration Status ===')
print(f'Interactive Static Dashboard (`dashboard/index.html`): {"EXISTS" if dashboard_index.exists() else "MISSING"}')
print(f'Streamlit Platform App (`platform/app.py`): {"EXISTS" if platform_app.exists() else "MISSING"}')
print('\nTo launch the Streamlit Platform App, execute:')
print('  streamlit run platform/app.py')
print('To serve the Interactive Static Dashboard, execute:')
print('  python -m http.server --directory dashboard 8000')

=== BarrierLens Platform Integration Status ===
Interactive Static Dashboard (`dashboard/index.html`): EXISTS
Streamlit Platform App (`platform/app.py`): EXISTS

To launch the Streamlit Platform App, execute:
  streamlit run platform/app.py
To serve the Interactive Static Dashboard, execute:
  python -m http.server --directory dashboard 8000
